# Imports

In [3]:
import os
import json
import itables
import pandas as pd
from helpers import *

# itables.init_notebook_mode(all_interactive=True)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Wave Question Extraction
Parse raw data sets to extract questions relevant to the research topic. Relevant questions ask respondants about either military aid for Ukraine or economic sanctions on Russia.
Additionally extract fieldwork timelines during which respondants were surveyed and confirm consistency in questions to have a stable DV over time.

In [2]:
collected = []

for file in os.listdir(EB_BASE_PATH):
    collected.append(parse_doc(file))

collected.sort(key=lambda k: k["wave_id"])
df = pd.DataFrame(collected)
df

,eb_n,wave_id,fw_start,fw_end,season,questions,q_ids
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"[QE2.1. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing supply and delivery of military equipment to Ukraine]","[QE2_1, QE2_3]"
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_3]"
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disa

In [3]:
if os.path.exists("data/interim/wave_questions.csv"):
    questions = pd.read_csv("data/interim/wave_questions.csv")
else:
    qs = pd.DataFrame(df["questions"].tolist(), index=df.index)
    qs.columns = ["q1", "q2"]
    qs["q1"] = qs["q1"].apply(lambda v: f"{v.split(" ", 1)[0]} {v.split(":-")[-1]}")
    qs["q2"] = qs["q2"].apply(lambda v: f"{v.split(" ", 1)[0]} {v.split(":-")[-1]}")
    questions = df.join(qs).drop(columns=["questions", "q_ids"])

    with open("data/interim/wave_questions.csv", "w") as f:
        questions.to_csv(f, index=False)

questions

,eb_n,wave_id,fw_start,fw_end,season,q1,q2
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing supply and delivery of military equipment to Ukraine
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.3. Financing the purchase and supply of military equipment to Ukraine
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
8,eb105,105.2,2026-03-12,2026-04-05,Spring 2026,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals (including use of immobilised Russian assets to finance support for Ukraine)",QD2.2. Financing the purchase and supply of military equipment to Ukraine


Looking at the extracted questions from each wave, a change in the wording of the questions only happens twice: *QE2.3* in wave *97.5* and *QD2.1* in wave *105.2*. However, given that the deviations in wording are relatively minor - *97.5-QE2.3* implies the same thing as *98.2-QE2.3* and *105.2-QD2.1* clarifies the scope of the sanctions - a consistent analysis over time can still be made.

# Eurobarometer Outcomes
Per-wave, per-question, per-country response scores extracted from the Eurobarometer datasets

In [4]:
if os.path.exists("data/interim/wave_scores.json"):
    with open("data/interim/wave_scores.json", "r") as f:
        data = json.load(f)
else:
    data = {}

    for row in df[["eb_n", "q_ids"]].itertuples():
        data[row.eb_n] = collect_scores(row.eb_n, row.q_ids)  # type: ignore[arg-type]

    with open("data/interim/wave_scores.json", "w") as f:
        json.dump(data, f, indent=4)

series = pd.Series({
    (wave, q, COUNTRY_CODES[country]): metrics
    for wave, questions in data.items()
    for q, countries in questions.items()
    for country, metrics in countries.items()
})

wdf = pd.DataFrame(series.tolist(), index=series.index)
wdf.index.names = ['wave', 'question', 'country']

pd.reset_option("display.max_rows")
wdf

total  total_agree  total_disagree  totally_agree  \
wave  question country                                                       
eb97  QE2_1    Belgium    1009          819             178            437   
               Bulgaria   1038          477             440            205   
               Czechia    1015          727             249            484   
               Denmark    1037          956              67            762   
               Germany    1507         1229             224            859   
...                        ...          ...             ...            ...   
eb105 QD2_2    Romania    1054          480             538             99   
               Slovenia   1009          368             603             79   
               Slovakia   1003          359             616            125   
               Finland    1009          882             103            502   
               Sweden     1026          953              62            699   

                         tend_to_agree  dont_know  tend_to_disagree  \
wave  question country                                                
eb97  QE2_1    Belgium             382         12               131   
               Bulgaria            272        121               218   
               Czechia             243         39               146   
               Denmark             195         14                52   
               Germany             370         54               149   
...                                ...        ...               ...   
eb105 QD2_2    Romania             381         37               309   
               Slovenia            289         37               360   
               Slovakia            234         29               310   
               Finland             380         24                76   
               Sweden              254         10                41   

                         totally_disagree  
wave  question country                     
eb97  QE2_1    Belgium                 47  
               Bulgaria               222  
               Czechia                103  
               Denmark                 15  
               Germany                 75  
...                                   ...  
eb105 QD2_2    Romania                229  
               Slovenia               243  
               Slovakia               306  
               Finland                 27  
               Sweden                  21  

[486 rows x 8 columns]

# Exposure Coding

Derive the 0-3 exposure tier for each country-window directly from the frozen source register and the codebook observation windows. Criteria A/B/C are hand-coded in the register; D (criterion A corroborated by >=2 independent sources) is computed here as >=2 distinct authors among the sources establishing A. This cell recomputes on every run rather than caching, so the index always reflects the current register (R9). It writes back only the derived columns of the "coding" sheet — manual columns (locators, "ambiguous", bounds, prose notes) and the other sheets are left intact.

In [5]:
register = pd.read_csv("data/interim/source_register.csv")
sources = load_sources(register)

EXPOSURE_XLSX = "data/interim/exposure_coding.xlsx"
coding = pd.read_excel(EXPOSURE_XLSX, sheet_name="coding")

codes = coding.apply(lambda r: code_country_window(
    r["iso2"],
    pd.to_datetime(r["window_start"]).date(),
    pd.to_datetime(r["window_end"]).date(),
    sources), axis=1, result_type="expand")

for col in ["criteria_met", "notes",
            "evidence_1_source", "evidence_2_source", "evidence_3_source"]:
    coding[col] = coding[col].astype("object")

coding["exposure"] = codes["exposure"].astype(int)
coding["criteria_met"] = codes["criteria_met"]

ev = codes["evidence_sources"].str.split(";")
for i in range(3):
    coding[f"evidence_{i+1}_source"] = ev.apply(
        lambda xs: xs[i] if isinstance(xs, list) and len(xs) > i and xs[i] else pd.NA)

# seed note where none has been hand-entered; never overwrite one
blank = coding["notes"].isna() | (coding["notes"].astype(str).str.strip() == "")
coding.loc[blank, "notes"] = codes.loc[blank, "notes"].replace("", pd.NA)

coding["n_sources_A"] = codes["n_sources_A"].astype(int)
coding["n_indep_A"] = codes["n_indep_A"].astype(int)  # the basis for D, stored for defensibility

with pd.ExcelWriter(EXPOSURE_XLSX, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as xl:
    coding.to_excel(xl, sheet_name="coding", index=False)

pd.reset_option("display.max_rows")
coding

,wave_num,eb_n,wave_id,country,iso2,window_start,window_end,exposure,criteria_met,evidence_1_source,evidence_1_locator,evidence_2_source,evidence_2_locator,evidence_3_source,evidence_3_locator,ambiguous,bound_lo,bound_hi,notes,n_sources_A,n_indep_A
0,1,eb97,97.5,Belgium,BE,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,eb97,97.5,Bulgaria,BG,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,eb97,97.5,Czechia,CZ,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,1,eb97,97.5,Denmark,DK,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,1,eb97,97.5,Germany,DE,2022-02-24 00:00:00,2022-07-17,3,ABCD,CHK25,NaN,DFR22,NaN,EDL22,NaN,NaN,NaN,NaN,"A:CHK25,DFR22,EDL22,ISD22,MET22,MET24-2 | B:DFR22,EDL22,ISD22,MET22,MET24-2 | C:CHK25,DFR22,EDL22,ISD22,MET22",6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,9,eb105,105.2,Romania,RO,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
239,9,eb105,105.2,Slovenia,SI,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
240,9,eb105,105.2,Slovakia,SK,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
241,9,eb105,105.2,Finland,FI,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [6]:
# truncation boundary: waves with no documented exposure anywhere -> drop them (R10)
wave_max = coding.groupby("wave_num")["exposure"].max()
usable = wave_max[wave_max > 0].index.tolist()
truncated = wave_max[wave_max == 0].index.tolist()

# R11 ceiling flag, computed on the truncated panel (all-zero tail masks it otherwise)
trunc = coding[coding["wave_num"].isin(usable)]
var = trunc.groupby("iso2")["exposure"].nunique()
flat = var[var == 1].index.tolist()

print("usable waves:", usable, "| truncate (all-zero):", truncated)
print("no within-country variation (drop out under country FE):", flat)
trunc.pivot(index="iso2", columns="wave_num", values="exposure")

usable waves: [1, 2, 3, 4, 5, 6, 7] | truncate (all-zero): [8, 9]
no within-country variation (drop out under country FE): ['BE', 'DE', 'LU', 'MT']


wave_num,1,2,3,4,5,6,7
iso2,,,,,,,
AT,0,0,0,2,2,0,0
BE,0,0,0,0,0,0,0
BG,0,0,0,0,1,0,0
CY,0,0,0,0,1,0,0
CZ,0,0,0,0,1,0,0
DE,3,3,3,3,3,3,3
DK,0,0,0,0,1,0,0
EE,1,1,0,0,1,0,0
EL,0,0,0,0,1,0,0


Populate source-to-waves crosswalk

In [7]:
waves = (coding[["wave_num", "window_start", "window_end"]]
         .drop_duplicates()
         .sort_values("wave_num"))
waves["window_start"] = pd.to_datetime(waves["window_start"]).dt.date
waves["window_end"] = pd.to_datetime(waves["window_end"]).dt.date

rows = []
for s in sources:
    a, b = s["window"]
    row = {"source_id": s["s_id"]}
    for w in waves.itertuples():
        overlap = bool(a and b and a <= w.window_end and b >= w.window_start)  # type: ignore[arg-type]
        row[f"W{w.wave_num}"] = ";".join(sorted(s["countries"])) if overlap else ""
    rows.append(row)

s2w = pd.DataFrame(rows, columns=["source_id"] + [f"W{n}" for n in waves["wave_num"]])

with pd.ExcelWriter(EXPOSURE_XLSX, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as xl:
    s2w.to_excel(xl, sheet_name="source_to_waves", index=False)

s2w

,source_id,W1,W2,W3,W4,W5,W6,W7,W8,W9
0,EDL22,DE;EE;FR;IT;LT;LV,DE;EE;FR;IT;LT;LV,,,,,,,
1,DFR22,DE;FR;IT;LV;PL,DE;FR;IT;LV;PL,,,,,,,
2,MET22,DE;FR;IT;LV,DE;FR;IT;LV,,,,,,,
3,ISD22,DE;FR;IT,DE;FR;IT,,,,,,,
4,VIG23,,DE;FR,DE;FR,DE;FR,,,,,
5,RST23,,DE;FR;IT,DE;FR;IT,,,,,,
6,MET23,,,DE;FR;PL,DE;FR;PL,,,,,
7,VIG24,,,,AT;DE;ES;FR;PL,AT;DE;ES;FR;PL,,,,
8,AIF24,,,,DE;FR,DE;FR,,,,
9,AIF24-2,,,,,DE;FR;IT;PL,DE;FR;IT;PL,,,


# Controls (Eurostat)
GDP per capita (real), HICP inflation, and Ukrainian displaced-population share, matched to each wave's fieldwork midpoint. Also collects pop2021 for the PSM moderator.

In [13]:
codes = list(COUNTRY_CODES.keys())

if os.path.exists("data/interim/controls.csv") and os.path.exists("data/interim/pop2021.csv"):
    controls = pd.read_csv("data/interim/controls.csv")
    pop2021  = pd.read_csv("data/interim/pop2021.csv").set_index("iso2")["population"].to_dict()
else:
    # wave reference: fieldwork midpoint -> (year, YYYY-MM)
    wr = questions[["eb_n","fw_start","fw_end"]].copy()
    wr[["fw_start","fw_end"]] = wr[["fw_start","fw_end"]].apply(pd.to_datetime)
    wr["mid"]  = wr.fw_start + (wr.fw_end - wr.fw_start)/2
    wr["year"] = wr.mid.dt.year
    wr["ym"]   = wr.mid.dt.strftime("%Y-%m")
    wr = wr.merge(coding[["eb_n","wave_num"]].drop_duplicates(), on="eb_n")

    # GDP per capita, real (chain-linked volume, EUR per head), annual
    g = eurostat_long("nama_10_pc", geos=codes, dims={"na_item": "B1GQ"}, since="2021")
    clv = sorted(u for u in g["unit"].unique() if u.startswith("CLV") and u.endswith("_EUR_HAB"))
    g = g[g["unit"] == clv[-1]]
    gdp = g.assign(year=lambda d: d.period.astype(int)).rename(columns={"value":"gdp_pc"})[
          ["geo","year","gdp_pc"]]

    # HICP, monthly annual rate of change, all-items
    h = eurostat_long("prc_hicp_manr", geos=codes, dims={"coicop": "CP00"}, since="2022-01")
    hicp = h.rename(columns={"period":"ym","value":"hicp"})[["geo","ym","hicp"]]

    # Ukrainian temporary-protection beneficiaries (monthly stock)
    b = eurostat_long("migr_asytpsm", geos=codes,
                      dims={"citizen":"UA","sex":"T","age":"TOTAL"}, since="2022-01")
    benef = b.rename(columns={"period":"ym","value":"benef"})[["geo","ym","benef"]]

    # population (annual, 1 Jan)
    p = eurostat_long("demo_pjan", geos=codes, dims={"sex":"T","age":"TOTAL"}, since="2021")
    pop = p.assign(year=lambda d: d.period.astype(int)).rename(columns={"value":"population"})[
          ["geo","year","population"]]

    # assemble per (country, wave)
    base = pd.DataFrame({"geo": codes}).merge(wr[["wave_num","year","ym"]], how="cross")
    controls = (base
        .merge(gdp,   on=["geo","year"], how="left")
        .merge(hicp,  on=["geo","ym"],   how="left")
        .merge(benef, on=["geo","ym"],   how="left")
        .merge(pop,   on=["geo","year"], how="left"))
    controls["displaced_share"] = 100 * controls["benef"].fillna(0) / controls["population"]
    controls = controls.rename(columns={"geo":"iso2"})[
        ["iso2","wave_num","gdp_pc","hicp","displaced_share"]]

    pop2021 = pop[pop.year==2021].set_index("geo")["population"].to_dict()

    controls.to_csv("data/interim/controls.csv", index=False)
    pd.Series(pop2021, name="population").rename_axis("iso2").to_csv("data/interim/pop2021.csv")

controls

,iso2,wave_num,gdp_pc,hicp,displaced_share
0,BE,1,43890.0,10.4,0.439290
1,BE,2,44190.0,7.4,0.520319
2,BE,3,44190.0,1.6,0.563111
3,BE,4,44190.0,-0.8,0.613866
4,BE,5,44410.0,4.9,0.651048
...,...,...,...,...,...
238,SE,5,48770.0,2.4,0.370367
239,SE,6,48770.0,1.6,0.425760
240,SE,7,49310.0,2.1,0.340442
241,SE,8,49310.0,3.1,0.452459
